# Why does the real-data detector fail at high SNR? (harmonic / spectral-leakage check)

In R1, the real-data detector (GED of 13–30 Hz against **5–11 + 32–45 Hz**, quality gate, beta envelope divided by
the 1/f fit over **2–6.5 + 32–45 Hz**) collapsed at +10 dB (F1 ≈ 0.2–0.45, error 36–77 mm). The original
broadband-reference GED did not, and the oracle (true burst times) stayed at ~12 mm, so the failure is in
**detection / component choice**.

Candidate causes, each switched on or off separately, on the **same simulated recordings** (same seeds as R1):

| Condition | Burst waveform | GED reference bands | 1/f fit flank | Tests |
|---|---|---|---|---|
| `baseline` | sharp troughs (0.3) | 5–11 + 32–45 | 2–6.5 + 32–45 | reproduces R1 |
| `sinusoidal` | sinusoidal (0) | 5–11 + 32–45 | 2–6.5 + 32–45 | 2nd harmonic of non-sinusoidal bursts (32–48 Hz) |
| `ged_ref_45_60` | sharp | 5–11 + **45–60** | 2–6.5 + 32–45 | harmonic/leakage energy in the GED **reference** |
| `ged_ref_and_flank_45_60` | sharp | 5–11 + **45–60** | 2–6.5 + **45–60** | ... and in the **1/f normalisation / gate score** |
| `broadband_ged` | sharp | broadband (original simulation) | 2–6.5 + 32–45 | positive control |

Mechanism diagnostics for every recording: which of the six GED components matches the true source topography
(|r| with the average-referenced gain vector), whether the gate/score chose that component, its eigenvalue rank, its
beta peak above 1/f, and the fraction of the true source's power that falls in each reference band.

**How to read the result**
* `sinusoidal` restores +10 dB → the cause is the **harmonic** of non-sinusoidal (sharp-trough) bursts.
* `ged_ref_45_60` restores it but `sinusoidal` does not → **spectral leakage of short bursts** (Hann-windowed bursts
  of ~0.25 s are broadband) into the 32–45 Hz reference.
* Only `ged_ref_and_flank_45_60` restores it → the **1/f normalisation and gate score** (beta peak measured against a
  fit that includes the burst's own energy).
* Nothing but `broadband_ged` restores it → the **flanking-band contrast itself**: at high SNR a narrowband background
  rhythm out-scores a strong but spectrally broader burst source.

Put this notebook next to `revision_analyses.ipynb` (it reuses its code and the cached head models).
Runtime: about 20 min with the defaults (San Diego, 6 seeds). Cached in `pd_beta_three/revision/H1_*.csv`.

In [1]:
"""Settings."""
H_DATASET = "sandiego"            # montage and recording length to simulate ("sandiego", "newmexico", "iowa")
H_SNRS = (0.0, 5.0, 10.0)         # 0 dB = where everything works; 5 and 10 dB = where the collapse appears
H_REGIONS = ["L_M1_hand", "R_M1_hand", "L_SMA"]
H_SEEDS = 6                       # per region × SNR × condition (paired across conditions); max 10 = R1's seeds
H_METHODS = ["dSPM", "LCMV"]
REVISION_NOTEBOOK = "revision_analyses.ipynb"

In [2]:
"""Load the code of the revision notebook (its settings, loader and R1 functions only; nothing heavy runs)."""
import json
from pathlib import Path

_nb = json.loads(Path(REVISION_NOTEBOOK).read_text(encoding="utf-8"))
_want = ('"""Settings."""', '"""Load every definition', "def sim_record(")
RESULTS_THR = {}
for _cell in _nb["cells"]:
    _src = "".join(_cell["source"]) if isinstance(_cell["source"], list) else _cell["source"]
    if _cell["cell_type"] == "code" and _src.lstrip().startswith(_want):
        exec(compile(_src, REVISION_NOTEBOOK, "exec"), globals())
assert H_DATASET in ANAT, f"{H_DATASET} not available: {list(ANAT)}"
_SPECTRAL_ORIG, _REF_ORIG = spectral, list(REF_BANDS)
print(f"Loaded. {LABEL[H_DATASET]}: {len(ANAT[H_DATASET]['ch_names'])} channels, {DUR[H_DATASET]:.0f} s per recording")

Loaded pd_beta_three_datasets.ipynb: datasets ['sandiego', 'newmexico', 'iowa'], head model fsaverage, outputs → pd_beta_three\revision
  San Diego: 32 channels, simulated duration 183 s
  New Mexico: 60 channels, simulated duration 58 s
  Iowa: 58 channels, simulated duration 132 s
Loaded. San Diego: 32 channels, 183 s per recording


In [3]:
def make_spectral(flank_hi=(32.0, 45.0), fmax=45.0):
    """spectral() of the main notebook with a configurable upper 1/f flank (identical to it for the defaults)."""
    def spectral_alt(sig, valid, fs):
        f, p = signal.welch(sig[valid], fs, nperseg=int(2 * fs))
        sel = (f >= 2) & (f <= fmax)
        f, p = f[sel], p[sel]
        flank = ((f >= 2) & (f <= 6.5)) | ((f >= flank_hi[0]) & (f <= flank_hi[1]))
        coef = np.polyfit(np.log10(f[flank]), np.log10(p[flank]), 1)
        fit = np.polyval(coef, np.log10(f))
        resid = 10 * (np.log10(p) - fit)
        b = (f >= 13) & (f <= 30)
        j = np.argmax(resid[b])
        trap = getattr(np, "trapezoid", None) or np.trapz
        return {"beta_db": float(resid[b][j]), "peak_hz": float(f[b][j]), "slope": float(coef[0]),
                "ap_var": float(trap(10 ** fit[b], f[b])), "curve": (f, 10 * np.log10(p), 10 * fit)}
    return spectral_alt


CONDITIONS = {
    "baseline":                {"sharp": CFG["burst_sharpness"], "ref": _REF_ORIG, "flank": (32.0, 45.0), "broadband": False},
    "sinusoidal":              {"sharp": 0.0, "ref": _REF_ORIG, "flank": (32.0, 45.0), "broadband": False},
    "ged_ref_45_60":           {"sharp": CFG["burst_sharpness"], "ref": [(5.0, 11.0), (45.0, 60.0)], "flank": (32.0, 45.0), "broadband": False},
    "ged_ref_and_flank_45_60": {"sharp": CFG["burst_sharpness"], "ref": [(5.0, 11.0), (45.0, 60.0)], "flank": (45.0, 60.0), "broadband": False},
    "broadband_ged":           {"sharp": CFG["burst_sharpness"], "ref": _REF_ORIG, "flank": (32.0, 45.0), "broadband": True},
}


def set_condition(c):
    """Swap the globals the real-data functions read (REF_BANDS in ged_components, spectral in evaluate/normalised_beta)."""
    global spectral
    REF_BANDS[:] = c["ref"]
    spectral = _SPECTRAL_ORIG if tuple(c["flank"]) == (32.0, 45.0) else make_spectral(c["flank"], 60.0)


def band_fraction(s, fs):
    """Power of the true source course in each band, relative to its 13–30 Hz power."""
    f, p = signal.welch(s, fs, nperseg=int(2 * fs))
    bp = lambda lo, hi: p[(f >= lo) & (f <= hi)].sum()
    beta = bp(13, 30)
    return {"src_5_11": bp(5, 11) / beta, "src_32_45": bp(32, 45) / beta, "src_45_60": bp(45, 60) / beta}


def seed_of(region, snr, k):
    si = list(R1_SNRS).index(snr) if snr in R1_SNRS else 50 + int(snr)
    ri = R1_REGIONS.index(region) if region in R1_REGIONS else 90 + H_REGIONS.index(region)
    return 700_000 + 1000 * ri + 10 * si + k


def run_one(cond_name, region, snr, k):
    c = CONDITIONS[cond_name]
    set_condition(c)
    a = ANAT[H_DATASET]
    cfg = {**CFG, "duration": DUR[H_DATASET], "burst_sharpness": c["sharp"]}
    seed = seed_of(region, snr, k)
    X, s, ev = sim.simulate(a, region, "OFF", float(snr), seed, cfg)
    v = np.ones(X.shape[1], bool)
    pad = int(EDGE_PAD_S * SFREQ)
    v[:pad] = v[-pad:] = False
    g = a["regions"][region]["gain"]
    g = g - g.mean()
    # all six candidate components, as choose_detector() sees them
    W, A = ged_components([X], [v], SFREQ)
    cands = [evaluate(W[:, i], A[:, i], [X], [v], SFREQ, a["ch_names"]) for i in range(W.shape[1])]
    match = np.array([abs(np.corrcoef(A[:, i], g)[0, 1]) for i in range(W.shape[1])])
    ok = [i for i, cd in enumerate(cands) if cd["passed"]]
    chosen = max(ok or range(len(cands)), key=lambda i: cands[i]["score"])
    true_i = int(np.argmax(match))
    score_rank = int(np.argsort([-cd["score"] for cd in cands]).tolist().index(true_i)) + 1
    row = {"condition": cond_name, "dataset": H_DATASET, "region": region, "snr_db": snr, "seed": seed,
           "gate_passed": bool(ok), "chosen_component": chosen + 1, "true_component": true_i + 1,
           "chose_true": chosen == true_i, "match_chosen": match[chosen], "match_best": match[true_i],
           "true_eig_rank": true_i + 1, "true_score_rank": score_rank, "true_beta_db": cands[true_i]["beta_db"],
           "true_sm_frac": cands[true_i]["sm_frac"], "true_passed": cands[true_i]["passed"],
           "chosen_beta_db": cands[chosen]["beta_db"], **band_fraction(s, SFREQ)}
    if c["broadband"]:                                         # original simulation's filter, no gate
        w = sim.ged_filter(X, SFREQ, CFG)
        pat = np.cov(sim.bandpass(X, CFG["beta_band"], SFREQ)) @ w
        row.update({"match_chosen": abs(np.corrcoef(pat, g)[0, 1]), "chose_true": np.nan, "gate_passed": True})
    else:
        w = W[:, chosen]
    sb, _ = normalised_beta(X, v, w, SFREQ)
    thr = np.percentile(np.abs(signal.hilbert(sb))[v], THRESH_PERCENTILE)
    out = []
    for r in score_run(a, X, v, ev, region, sb, thr, cfg, methods=H_METHODS, oracle=False):
        out.append({**row, **{k_: r.get(k_, np.nan) for k_ in ("method", "f1", "precision", "recall", "peak_err_mm",
                                                                 "region_correct", "nave")}})
    return out

In [4]:
f = REV / f"H1_harmonic_check_{H_DATASET}.csv"
if f.exists():
    H = pd.read_csv(f)
else:
    jobs = [(cn, reg, snr, k) for snr in H_SNRS for reg in H_REGIONS for k in range(H_SEEDS) for cn in CONDITIONS]
    rows, t0 = [], time.time()
    try:
        for i, job in enumerate(jobs):
            rows += run_one(*job)
            el = time.time() - t0
            print(f"\r  {i + 1}/{len(jobs)}  ({el:.0f} s, ~{el / (i + 1) * (len(jobs) - i - 1):.0f} s left)", end="")
    finally:
        set_condition(CONDITIONS["baseline"])                  # restore the main notebook's settings
    print()
    H = pd.DataFrame(rows)
    H.to_csv(f, index=False)

# sanity check: the baseline must reproduce R1 (same seeds, per-recording threshold)
r1f = REV / f"R1_sim_real_detector_{H_DATASET}.csv"
if r1f.exists():
    r1 = pd.read_csv(r1f)
    r1 = r1[(r1.scope == "per_recording") & r1.method.isin(H_METHODS)][["seed", "method", "peak_err_mm", "f1"]]
    chk = H[H.condition == "baseline"].merge(r1, on=["seed", "method"], suffixes=("", "_R1"))
    if len(chk):
        print(f"Baseline vs R1 (same seeds): {len(chk)} maps, max |Δ error| = "
              f"{np.nanmax(np.abs(chk.peak_err_mm - chk.peak_err_mm_R1)):.2f} mm, "
              f"max |Δ F1| = {np.nanmax(np.abs(chk.f1 - chk.f1_R1)):.3f}  (0 = identical)")

  270/270  (1084 s, ~0 s left))
Baseline vs R1 (same seeds): 108 maps, max |Δ error| = 0.00 mm, max |Δ F1| = 0.000  (0 = identical)


In [5]:
"""Results."""
order = list(CONDITIONS)
d = H.copy()
d["condition"] = pd.Categorical(d.condition, order, ordered=True)
summ = d.groupby(["condition", "snr_db", "method"], observed=True).agg(
    n=("seed", "size"), F1=("f1", "mean"), err_mm=("peak_err_mm", "median"),
    region_acc=("region_correct", "mean"), gate_pass=("gate_passed", "mean")).reset_index()
summ = summ.pivot_table(index=["condition", "snr_db"], columns="method",
                        values=["err_mm", "region_acc"], observed=True).round(2)
summ.columns = [f"{a}_{b}" for a, b in summ.columns]
det = d[d.method == H_METHODS[0]].groupby(["condition", "snr_db"], observed=True).agg(
    F1=("f1", "mean"), gate_pass=("gate_passed", "mean"), chose_true=("chose_true", "mean"),
    match_chosen=("match_chosen", "median"), true_eig_rank=("true_eig_rank", "median"),
    true_score_rank=("true_score_rank", "median"), true_beta_db=("true_beta_db", "median"),
    true_passed=("true_passed", "mean")).round(2)
T = det.join(summ).reset_index()
save(T, f"H1_summary_{H_DATASET}")
print(f"{LABEL[H_DATASET]} montage, {H_SEEDS} seeds × {len(H_REGIONS)} regions per cell. "
      "chose_true = gate/score picked the component that matches the true topography; "
      "match_chosen = |r| of the chosen pattern with the true gain; ranks are among the 6 GED components (1 = top).")
print(T.to_string(index=False))

src = d[(d.method == H_METHODS[0]) & d.condition.isin(["baseline", "sinusoidal"])].groupby(
    "condition", observed=True)[["src_5_11", "src_32_45", "src_45_60"]].median()
print("\nTrue burst source: power in each band relative to its 13–30 Hz power (median over recordings):")
print(src.round(4).to_string())
save(src.reset_index(), f"H1_source_spectrum_{H_DATASET}")

# paired test at the highest SNR: does each manipulation reduce dSPM error relative to baseline?
top = max(H_SNRS)
b = H[(H.condition == "baseline") & (H.snr_db == top)].set_index(["seed", "method"])
tests = []
for cn in order[1:]:
    x = H[(H.condition == cn) & (H.snr_db == top)].set_index(["seed", "method"])
    for m in H_METHODS:
        both = b.xs(m, level="method").index.intersection(x.xs(m, level="method").index)
        e0 = b.xs(m, level="method").loc[both, "peak_err_mm"]
        e1 = x.xs(m, level="method").loc[both, "peak_err_mm"]
        f0 = b.xs(m, level="method").loc[both, "f1"]
        f1_ = x.xs(m, level="method").loc[both, "f1"]
        ok = e0.notna() & e1.notna()
        p = stats.wilcoxon(e0[ok], e1[ok]).pvalue if ok.sum() > 5 and np.any(e0[ok] != e1[ok]) else np.nan
        tests.append({"condition": cn, "method": m, "snr_db": top, "n": int(ok.sum()),
                      "baseline_err_mm": e0[ok].median(), "condition_err_mm": e1[ok].median(),
                      "baseline_F1": f0.mean(), "condition_F1": f1_.mean(), "wilcoxon_p": p})
print(f"\nPaired comparison with baseline at {top:+.0f} dB (same simulated recordings):")
print(save(pd.DataFrame(tests), f"H1_paired_{H_DATASET}").round(3).to_string(index=False))

San Diego montage, 6 seeds × 3 regions per cell. chose_true = gate/score picked the component that matches the true topography; match_chosen = |r| of the chosen pattern with the true gain; ranks are among the 6 GED components (1 = top).
              condition  snr_db   F1  gate_pass chose_true  match_chosen  true_eig_rank  true_score_rank  true_beta_db  true_passed  err_mm_LCMV  err_mm_dSPM  region_acc_LCMV  region_acc_dSPM
               baseline     0.0 0.73       1.00        1.0          0.94            1.0              1.0          5.55         1.00        12.92        12.92             0.83             0.94
               baseline     5.0 0.86       1.00        1.0          1.00            1.0              1.0          8.78         1.00         8.63        10.92             1.00             0.94
               baseline    10.0 0.42       0.89   0.333333          0.51            1.0              1.0         11.05         0.22        51.00        54.96             0.39             